## Import Libraries

In [33]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
from pathlib import Path

### Tải dữ liệu và xử lý lỗi font chữ

In [34]:
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
)
DATA_DIR = PROJECT_ROOT / 'data'

# Read dataset
raw_path = DATA_DIR / 'raw_data.csv'
df = pd.read_csv(raw_path, encoding='cp1252')

print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]:,}')
display(df.head())

Rows: 445
Columns: 7


,City,Country,Venue,Opening act(s),Attendance (tickets sold / available),Revenue,Tour
0,Evansville,United States,Roberts Municipal Stadium,Gloriana\r\nKellie Pickler,"7,463 / 7,463","$360,617",Fearless_Tour
1,Jonesboro,United States,Convocation Center,Gloriana\r\nKellie Pickler,"7,822 / 7,822","$340,328",Fearless_Tour
2,St. Louis,United States,Scottrade Center,Gloriana\r\nKellie Pickler,"13,764 / 13,764","$650,420",Fearless_Tour
3,Alexandria,United States,Bishop Ireton High School,Gloriana\r\nKellie Pickler,—,—,Fearless_Tour
4,North Charleston,United States,North Charleston Coliseum,Gloriana\r\nKellie Pickler,"8,751 / 8,751","$398,154",Fearless_Tour


### Tách cột 'Attendance' thành 2 cột 'Tickets sold' và 'Tickets available'

In [35]:
df[['Tickets sold', 'Tickets available']] = df['Attendance (tickets sold / available)'].str.split('/', expand=True)

### Làm sạch ký tự đặc biệt và ép kiểu dữ liệu

#### Bỏ cột Attendance cũ

In [36]:
df = df.drop('Attendance (tickets sold / available)', axis=1)


#### Xóa ký tự '$' và ','

In [37]:
df['Revenue'] = df['Revenue'].str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['Tickets sold'] = df['Tickets sold'].str.replace(',', '', regex=False)
df['Tickets available'] = df['Tickets available'].str.replace(',', '', regex=False)

#### Ép kiểu về số (Numeric)

In [38]:
df['Revenue'] = pd.to_numeric(df['Revenue'], errors='coerce')
df['Tickets sold'] = pd.to_numeric(df['Tickets sold'], errors='coerce')
df['Tickets available'] = pd.to_numeric(df['Tickets available'], errors='coerce')

#### Tính toán thêm cột 'Tickets remaining'

In [39]:
df['Tickets remaining'] = df['Tickets available'] - df['Tickets sold']

#### Làm sạch cột 'Opening act(s)' 

In [40]:
df['Opening act(s)'] = df['Opening act(s)'].fillna('No Opening Act')
df['Opening act(s)'] = df['Opening act(s)'].replace('—', 'No Opening Act')

#### Xóa các ký tự xuống dòng và ký tự lỗi nếu có

In [41]:
df['Opening act(s)'] = df['Opening act(s)'].str.replace('\r\n', ' and ', regex=False).str.replace('&', ' and ', regex=False)

#### Làm sạch cột Tour: Xóa dấu gạch dưới và thay bằng khoảng trắng


In [42]:
df['Tour'] = df['Tour'].str.replace('_', ' ', regex=False)

#### Xử lý giá trị khuyết thiếu (Missing Values) bằng cách điền giá trị trung bình (Mean)

In [43]:
df.dropna(subset=['Revenue'], inplace=True)

#### Xử lý dữ liệu bị trùng lặp

In [44]:
# Xác định các cột chứa giá trị gộp cần chia đều
cols_to_divide = ['Revenue', 'Tickets sold', 'Tickets available']

# NÂNG CẤP: Đếm số show dùng CHUNG một cục Doanh thu & Lượng vé báo cáo gộp
df['Show_Count'] = df.groupby(['Tour', 'Revenue', 'Tickets sold'])['Revenue'].transform('count')

# Chia đều doanh thu và vé cho số show đang xài chung cục data đó
for col in cols_to_divide:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col] / df['Show_Count']

# Xóa cột đếm
df.drop(columns=['Show_Count'], inplace=True)

#### Đếm số nghệ sĩ mở màn

In [45]:
df['Number of opening acts'] = np.where(df['Opening act(s)'] == 'No Opening Act', 0, 
df['Opening act(s)'].str.count(' and ') + 1)

In [46]:
# Tạo 2 biến mới phục vụ phân tích Business Insights
df['Capacity utilization'] = df['Tickets sold'] / df['Tickets available']
df['Average ticket price'] = df['Revenue'] / df['Tickets sold']

#### Xem kết quả

In [47]:
display(df.describe())

,Revenue,Tickets sold,Tickets available,Tickets remaining,Number of opening acts,Capacity utilization,Average ticket price
count,4.060000e+02,406.000000,406.000000,406.000000,406.000000,406.000000,406.000000
mean,2.287708e+06,23470.674877,23517.980296,47.305419,1.564039,0.996733,87.946462
std,2.227541e+06,18032.454928,18035.510712,590.900263,0.662645,0.036692,28.570913
min,1.533030e+05,3394.500000,3394.500000,0.000000,0.000000,0.423294,34.533061
25%,8.691135e+05,12688.333333,12688.333333,0.000000,1.000000,1.000000,65.792600
50%,1.135099e+06,13938.500000,13938.500000,0.000000,2.000000,1.000000,81.346791
75%,3.473851e+06,38837.500000,39818.250000,0.000000,2.000000,1.000000,108.151583
max,9.350275e+06,75980.000000,75980.000000,10590.000000,3.000000,1.000000,182.223001


In [48]:
print(df[['Opening act(s)', 'Number of opening acts']].head(30))

                 Opening act(s)  Number of opening acts
0   Gloriana and Kellie Pickler                       2
1   Gloriana and Kellie Pickler                       2
2   Gloriana and Kellie Pickler                       2
4   Gloriana and Kellie Pickler                       2
5   Gloriana and Kellie Pickler                       2
6   Gloriana and Kellie Pickler                       2
7                No Opening Act                       0
8                No Opening Act                       0
9   Gloriana and Kellie Pickler                       2
10  Gloriana and Kellie Pickler                       2
11  Gloriana and Kellie Pickler                       2
12  Gloriana and Kellie Pickler                       2
13  Gloriana and Kellie Pickler                       2
14  Gloriana and Kellie Pickler                       2
15  Gloriana and Kellie Pickler                       2
16  Gloriana and Kellie Pickler                       2
17  Gloriana and Kellie Pickler                 